# NFO self-regulation Specific Visualization

In [ ]:
import os
os.chdir("../")

In [ ]:
import glob
import gzip
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from modelproblem import ModelProblem
from petab.visualize import plot_problem
from weighted_quantile import weighted_quantile
from result_classes import Result,MethodResults
import scipy.cluster.hierarchy as hier
from sklearn.cluster import AgglomerativeClustering


In [ ]:
prob_name = "Neg_Feed_Oscillate"
methods = ["smc"]
colors = sns.color_palette("tab10", n_colors=len(methods))

mod_prob = ModelProblem(prob_name)
mod_prob.initialize()

grouped_results = [MethodResults(x) for x in methods]

for method, group_obj in zip(methods, grouped_results):
	result_dir = f"results/{prob_name}/{method}/"
	fnames = glob.glob(result_dir + "*.pkl")
	for fname in fnames:
		with gzip.open(fname, "rb") as f:
			results = pickle.load(f)
		result_obj = Result(results)
		group_obj.add_result(result_obj)	

In [ ]:
fixed_idxs = mod_prob.problem.x_fixed_indices
par_names = mod_prob.problem.x_names
x=np.array(par_names)
mask=np.full(len(par_names),True,dtype=bool)
mask[fixed_idxs]=False
fit_par_names=x[mask]

dummy_idx = -1

par_bounds = mod_prob.bounds
plt.figure(figsize=(6,4), dpi=300)
ratios = np.array([x.get_sampling_efficiency(par_bounds, dummy_idx) for x in grouped_results])
ratio_df = pd.DataFrame(columns=methods, data=ratios.T)
print(ratio_df)
sns.boxplot(ratio_df)#, size=2)
plt.xticks(range(len(methods)), [x.abbr for x in grouped_results])
plt.xlabel("Method"); plt.ylabel(f"Sampling Quality");

## Select best result for each method

In [ ]:
llhs = np.array([x.get_avg_llhs() for x in grouped_results])
best_runs = [np.argmax(x) for x in llhs]
best_results = [res.all_runs[best_idx] for best_idx, res in zip(best_runs, grouped_results)]

smc = best_results[1]
pmc = best_results[2]

smc_dict = smc.algo_specific_info
print(smc_dict.keys())

print(smc_dict["n_mcmc"])
print(smc_dict["effective_ss"])
print(smc_dict["acceptance"])

In [ ]:
fixed_idxs = mod_prob.problem.x_fixed_indices
par_names = mod_prob.problem.x_names
x=np.array(par_names)
mask=np.full(len(par_names),True,dtype=bool)
mask[fixed_idxs]=False
fit_par_names=x[mask]

par_bounds = mod_prob.bounds
histtype = "step"
alpha=1

plt.figure(figsize=(18,18), dpi=300)
for i, par_name in enumerate(fit_par_names): 
	plt.subplot(4,4,i+1)
	for j in range(len(best_results)):      
		cur_result = best_results[j]
		norm_ws = cur_result.posterior_weights
		plt.hist(cur_result.posterior_samples[:, i], lw=2, weights=norm_ws, color=colors[j], alpha=alpha,
			 cumulative=True, histtype="step", bins=50, label=cur_result.method) 
		plt.xlabel(par_name)
		plt.yticks([])
		plt.ylabel("Density")
		plt.margins(x=0)
	plt.legend()
plt.tight_layout()

In [ ]:
# Get the names of the estimated parameters
fixed_idxs = mod_prob.problem.x_fixed_indices
par_names = mod_prob.problem.x_names
x=np.array(par_names)
mask=np.full(len(par_names),True,dtype=bool)
mask[fixed_idxs]=False
fit_par_names=x[mask]

par_bounds = mod_prob.bounds
histtype = "bar"
alpha=0.5

plt.figure(figsize=(10,12), dpi=300)
for i, par_name in enumerate(fit_par_names): 
	plt.subplot(4,4,i+1)
	for j in range(len(best_results)):      
		cur_result = best_results[j]
		plt.hist(cur_result.posterior_samples[:, i], lw=2, weights=cur_result.posterior_weights, color=colors[j], alpha=alpha,
			 cumulative=False, histtype=histtype, bins=40, label=cur_result.method) 
		plt.xlabel(par_name)
		plt.yticks([])
		plt.ylabel("Density")
		plt.margins(x=0)
	plt.legend()
plt.tight_layout()

In [ ]:
import scipy.stats as st
# Get the names of the estimated parameters
fixed_idxs = mod_prob.problem.x_fixed_indices
par_names = mod_prob.problem.x_names
x=np.array(par_names)
mask=np.full(len(par_names),True,dtype=bool)
mask[fixed_idxs]=False
fit_par_names=x[mask]

par_bounds = mod_prob.bounds
histtype = "bar"
alpha=0.5
xtrue = mod_prob.petab_problem.get_x_nominal(fixed=False, scaled=False)
plt.figure(figsize=(16,16), dpi=300)
for i, par_name in enumerate(fit_par_names): 
	plt.subplot(4,4,i+1)
	for j in range(len(best_results)):      
		cur_result = best_results[j]
		param_samples = cur_result.posterior_samples[:, i]
		norm_ws = cur_result.posterior_weights
		kde = st.gaussian_kde(param_samples, weights=norm_ws)
		x = np.linspace(np.min(param_samples), np.max(param_samples), 50)
		plt.plot(x, kde(x), '-', color=colors[j], alpha=0.75, zorder=1, label=cur_result.method)
		plt.fill_between(x, kde(x), alpha=0.25, color=colors[j], zorder=1)
	
	plt.axvline(x=xtrue[i], ls="--", color="k", label="Nominal")
	plt.xlabel(par_name)
	plt.yticks([])
	plt.ylabel("Density")
	#plt.margins(x=0, y=0.001)
	plt.legend()
plt.tight_layout()

In [ ]:
import corner
for i, method in enumerate(methods):
	cur_result = best_results[j]
	param_samples = cur_result.posterior_samples
	norm_ws = cur_result.posterior_weights #np.divide(cur_result.posterior_weights, np.sum(cur_result.posterior_weights))
	corner.corner(param_samples, weights=norm_ws, color=colors[i], labels=fit_par_names)
	

In [ ]:
from pypesto.objective import AggregatedObjective
from pypesto.objective.roadrunner.road_runner import RoadRunnerObjective
obj = mod_prob.problem.objective
og_obj = obj
if isinstance(obj, AggregatedObjective):
    subobjs = mod_prob.problem.objective.__dict__["_objectives"]
    for subobj in subobjs:
        if isinstance(subobj, RoadRunnerObjective):
            obj = subobj
        else:
            continue

## Plot fits to model data

In [ ]:
CI = 0.95
UPPER_PCT = (1 - (1-CI)/2)
LOWER_PCT = ((1-CI)/2)

petab_prob = mod_prob.petab_problem
ax_dict = plot_problem(petab_problem=petab_prob,) 
fig = plt.gcf()
# Change the figure size
fig.set_size_inches(6,4)

In [ ]:
measure_df = petab_prob.measurement_df
species = measure_df["observableId"].unique()
data_ts = measure_df["time"].unique()

n_sim_ts = int(36*100 + 1)
simu = obj.roadrunner_instance
simu.resetAll()
simu.timeCourseSelections = ["time", "S1", "S3", "S5"]
og_sim = simu.simulate(0, 36, n_sim_ts)
og_sim_ts = og_sim[:, 0]

fig, axs = plt.subplots(1,3,figsize=(18,4), dpi=300)
axs[0].plot(og_sim_ts, og_sim[:, 1], "k-", lw=0.5, label="True")
axs[0].plot(data_ts, measure_df[measure_df["observableId"]==species[0]]["measurement"], "ko", label="Data", zorder=2)

axs[1].plot(og_sim_ts, og_sim[:, 2], "k-", lw=0.5, label="True")
axs[1].plot(data_ts, measure_df[measure_df["observableId"]==species[1]]["measurement"], "ko", label="Data", zorder=2)

axs[2].plot(og_sim_ts, og_sim[:, 3], "k-", lw=0.5, label="True")
axs[2].plot(data_ts, measure_df[measure_df["observableId"]==species[2]]["measurement"], "ko", label="Data", zorder=2)

lss = [(0,(5,10)), (0,(3,5,1,5)), (0,(1,4))]
all_methods_sim_data = []

for i, best in enumerate(best_results):
	pars = best.posterior_samples
	weights = best.posterior_weights

	all_sim_data = np.empty(shape=(n_sim_ts, pars.shape[0], len(species)))

	## Collect all of the runs simulation information
	for n, par in enumerate(pars):
		#sim = obj(par, mode="mode_fun", return_dict=True)["simulation_results"]["simCondition"]
		simu.resetAll()
		simu.timeCourseSelections = ["time", "S1", "S3", "S5"]
		for name, x in zip(fit_par_names, par):
			simu[name] = x
		simu.reset()
		sim = simu.simulate(0, 36, n_sim_ts)
		all_sim_data[:, n, :] = sim[:, 1:]
		sim_ts = sim[:, 0]

	mean_sim_data = np.average(all_sim_data, weights=weights, axis=1)
	all_methods_sim_data.append(all_sim_data)
	temp1 = np.array([weighted_quantile(x[:, 0], [LOWER_PCT, UPPER_PCT], weights) for x in all_sim_data])
	temp2 = np.array([weighted_quantile(x[:, 1], [LOWER_PCT, UPPER_PCT], weights) for x in all_sim_data])
	temp3 = np.array([weighted_quantile(x[:, 2], [LOWER_PCT, UPPER_PCT], weights) for x in all_sim_data])

	axs[0].plot(sim_ts, mean_sim_data[:, 0], color=colors[i], linestyle=lss[i],
			  lw=2, label=grouped_results[i].abbr)
	axs[0].fill_between(sim_ts, temp1[:, 0], temp1[:, 1], zorder=3, alpha=0.3, color=colors[i])

	axs[1].plot(sim_ts, mean_sim_data[:, 1], color=colors[i], linestyle=lss[i],
			  lw=2, label=grouped_results[i].abbr)
	axs[1].fill_between(sim_ts, temp2[:, 0], temp2[:, 1], zorder=3, alpha=0.3, color=colors[i])

	axs[2].plot(sim_ts, mean_sim_data[:, 2], color=colors[i], linestyle=lss[i],
			  lw=2, label=grouped_results[i].abbr)
	axs[2].fill_between(sim_ts, temp3[:, 0], temp3[:, 1], zorder=3, alpha=0.3, color=colors[i])

axs[0].set_xlabel("Time")
axs[0].set_ylabel("XT")
axs[0].margins(x=0.02);
axs[1].set_xlabel("Time")
axs[1].set_ylabel("YP  Y(c~P)")
axs[1].margins(x=0.02);
axs[2].set_xlabel("Time")
axs[2].set_ylabel("RP  R(c~P)")
axs[2].legend(loc="upper right")
axs[2].margins(x=0.02);

## Cluster parameter sets

In [ ]:
# Taken from https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html#sphx-glr-auto-examples-cluster-plot-agglomerative-dendrogram-py
def plot_dendrogram(model, **kwargs):
	# Create linkage matrix and then plot the dendrogram
	# create the counts of samples under each node
	counts = np.zeros(model.children_.shape[0])
	n_samples = len(model.labels_)
	for i, merge in enumerate(model.children_):
		current_count = 0
		for child_idx in merge:
			if child_idx < n_samples:
				current_count += 1  # leaf node
			else:
				current_count += counts[child_idx - n_samples]
		counts[i] = current_count

	linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
   
	plt.figure(figsize=(16,7), dpi=600)
	#hier.set_link_color_palette(['#0632BF','#CD52E8', '#0EDA1C'])
	
	# Make the dendrogram and give the colour above threshold
	hier.dendrogram(linkage_matrix, **kwargs)
	#color_threshold=1e8, above_threshold_color='k', 
	#				leaf_font_size=12, leaf_rotation=90, **kwargs)
			
	plt.yticks([])
	plt.xticks([])
	plt.xlabel("Clustered Posterior Samples")
	plt.ylabel("Distance")
	#plt.title("Hierarchical Clustering\nWard Criterion, Euclidean Distance")
	plt.show()

### Cluster by simulation data

In [ ]:
for res, sim_data in zip(best_results, all_methods_sim_data): 
	# down sample sim_data
	trim_sim_data = np.swapaxes(sim_data[::10, :], 0, 1)
	print(trim_sim_data.shape)
	trim_sim_data = trim_sim_data.reshape((trim_sim_data.shape[0], 
														trim_sim_data.shape[1]*trim_sim_data.shape[2]))
	agg_model = AgglomerativeClustering(distance_threshold=0, n_clusters=None)
	agg_model = agg_model.fit(trim_sim_data)
	plot_dendrogram(agg_model)

In [ ]:
N_CLUSTERS = 6
colors = sns.color_palette("hsv", n_colors=N_CLUSTERS)
for res, sim_data, z in zip(best_results, all_methods_sim_data, range(3)): 
	# down sample sim_data
	trim_sim_data = sim_data[::10, :, 1]
	agg_model = AgglomerativeClustering(n_clusters=N_CLUSTERS)
	agg_model = agg_model.fit(trim_sim_data)
	labels = agg_model.labels_

	clus_sims = [sim_data[:, np.where(labels==i)[0], :] for i in range(N_CLUSTERS)]
	weights = res.posterior_weights
	clus_all_ws = [weights[np.where(labels==i)[0]] for i in range(N_CLUSTERS)]

	for i, (clus_sim, clus_ws) in enumerate(zip(clus_sims, clus_all_ws)):
		fig, axs = plt.subplots(1,3,figsize=(18,4), dpi=300)

		print(clus_sim.shape, clus_ws.shape)
		mean_sim_data = np.average(clus_sim, weights=clus_ws, axis=1)
		temp1 = np.array([weighted_quantile(x[:, 0], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])
		temp2 = np.array([weighted_quantile(x[:, 1], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])
		temp3 = np.array([weighted_quantile(x[:, 2], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])

		axs[0].plot(sim_ts, mean_sim_data[:, 0], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[0].fill_between(sim_ts, temp1[:, 0], temp1[:, 1], zorder=3, alpha=0.3, color=colors[i])

		axs[1].plot(sim_ts, mean_sim_data[:, 1], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[1].fill_between(sim_ts, temp2[:, 0], temp2[:, 1], zorder=3, alpha=0.3, color=colors[i])

		axs[2].plot(sim_ts, mean_sim_data[:, 2], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[2].fill_between(sim_ts, temp3[:, 0], temp3[:, 1], zorder=3, alpha=0.3, color=colors[i])

	axs[0].set_xlabel("Time")
	axs[0].set_ylabel("XT")
	axs[0].margins(x=0.02);
	axs[1].set_xlabel("Time")
	axs[1].set_ylabel("YP  Y(c~P)")
	axs[1].margins(x=0.02);
	axs[2].set_xlabel("Time")
	axs[2].set_ylabel("RP  R(c~P)")
	axs[2].legend(loc="upper right")
	axs[2].margins(x=0.02);
	fig.suptitle(grouped_results[z].abbr)

In [ ]:
param_idx = [4]
for res, sim_data in zip(best_results, all_methods_sim_data): 
	samples = res.posterior_samples[:, param_idx]
	agg_model = AgglomerativeClustering(distance_threshold=0, n_clusters=None)
	agg_model = agg_model.fit(samples)
	plot_dendrogram(agg_model)

In [ ]:
N_CLUSTERS = 5
from sklearn.mixture import GaussianMixture

colors = sns.color_palette("hsv", n_colors=N_CLUSTERS)
for res, sim_data, z in zip(best_results, all_methods_sim_data, range(3)): 
	samples = res.posterior_samples[:, 8]
	#agg_model = AgglomerativeClustering(n_clusters=N_CLUSTERS)
	#agg_model = agg_model.fit(samples)
	#labels = agg_model.labels_
	gmm = GaussianMixture(n_components=N_CLUSTERS)
	gmm.fit(samples.reshape(-1, 1))
	labels= gmm.predict(samples.reshape(-1, 1))

	clus_sims = [sim_data[:, np.where(labels==i)[0], :] for i in range(N_CLUSTERS)]
	weights = res.posterior_weights
	clus_all_ws = [weights[np.where(labels==i)[0]] for i in range(N_CLUSTERS)]

	for i, (clus_sim, clus_ws) in enumerate(zip(clus_sims, clus_all_ws)):
		fig, axs = plt.subplots(1,3,figsize=(18,4), dpi=300)

		print(clus_sim.shape, clus_ws.shape)
		mean_sim_data = np.average(clus_sim, weights=clus_ws, axis=1)
		temp1 = np.array([weighted_quantile(x[:, 0], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])
		temp2 = np.array([weighted_quantile(x[:, 1], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])
		temp3 = np.array([weighted_quantile(x[:, 2], [LOWER_PCT, UPPER_PCT], clus_ws) for x in clus_sim])

		axs[0].plot(sim_ts, mean_sim_data[:, 0], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[0].fill_between(sim_ts, temp1[:, 0], temp1[:, 1], zorder=3, alpha=0.3, color=colors[i])

		axs[1].plot(sim_ts, mean_sim_data[:, 1], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[1].fill_between(sim_ts, temp2[:, 0], temp2[:, 1], zorder=3, alpha=0.3, color=colors[i])

		axs[2].plot(sim_ts, mean_sim_data[:, 2], color=colors[i], lw=2, label=f"Cluster {i}")
		axs[2].fill_between(sim_ts, temp3[:, 0], temp3[:, 1], zorder=3, alpha=0.3, color=colors[i])

	axs[0].set_xlabel("Time")
	axs[0].set_ylabel("XT")
	axs[0].margins(x=0.02);
	axs[1].set_xlabel("Time")
	axs[1].set_ylabel("YP  Y(c~P)")
	axs[1].margins(x=0.02);
	axs[2].set_xlabel("Time")
	axs[2].set_ylabel("RP  R(c~P)")
	axs[2].legend(loc="upper right")
	axs[2].margins(x=0.02);
	fig.suptitle(grouped_results[z].abbr)

In [ ]:
measure_df = petab_prob.measurement_df
species = measure_df["observableId"].unique()
data_ts = measure_df["time"].unique()

for i, best in enumerate(best_results):
	pars = best.posterior_samples
	llhs = best.posterior_llhs
	weights = best.posterior_weights

	fig, axs = plt.subplots(1,3,figsize=(18,4), dpi=300)
	# Plot data
	axs[0].plot(data_ts, measure_df[measure_df["observableId"]==species[0]]["measurement"], "ko", label="Data", zorder=2)
	axs[1].plot(data_ts, measure_df[measure_df["observableId"]==species[1]]["measurement"], "ko", label="Data", zorder=2)
	axs[2].plot(data_ts, measure_df[measure_df["observableId"]==species[2]]["measurement"], "ko", label="Data", zorder=2)


	## Collect all of the runs simulation information
	n_good = 0
	for n, par in enumerate(pars):

		if llhs[n] > -21:
			n_good += 1
			simu.resetAll()
			simu.timeCourseSelections = ["time", "S1", "S3", "S5"]
			for name, x in zip(fit_par_names, par):
				simu[name] = x
			simu.reset()
			sim = simu.simulate(0, 36, n_sim_ts)
			sim_ts = sim[:, 0]

			axs[0].plot(sim_ts, sim[:, 1], lw=0.5, zorder=1)
			axs[1].plot(sim_ts, sim[:, 2], lw=0.5, zorder=1)
			axs[2].plot(sim_ts, sim[:, 3], lw=0.5, zorder=1)
	fig.suptitle(f"Num good fits = {n_good}")
	axs[0].set_xlabel("Time")
	axs[0].set_ylabel("XT")
	axs[0].margins(x=0.0);
	axs[1].set_xlabel("Time")
	axs[1].set_ylabel("YP  Y(c~P)")
	axs[1].margins(x=0.0);
	axs[2].set_xlabel("Time")
	axs[2].set_ylabel("RP  R(c~P)")
	axs[2].legend(loc="upper right")
	axs[2].margins(x=0.0);
	plt.show();